<a href="https://www.kaggle.com/code/syedasimalishah/build-your-own-jev-part-1-noul-yes-no?scriptVersionId=363499756" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🧠 Build Your Own Jev – Part 1: Noul (Yes/No)

**Series:** 👉 **Part 1: Noul** | [Part 2: Choice](https://www.kaggle.com/code/syedasimalishah/build-your-own-jev-part-2-choice-pick-one) | [Part 3: Score](https://www.kaggle.com/code/syedasimalishah/build-your-own-jev-part-3-score-rate-on-scale)

## What is this?
Jev is a "decision model". It does not write text. It gives answers your code can use directly.
In this series, we **build the same idea from scratch** with a free open model.
No API key needed.

## What you will learn
How to ask an llm a **yes/no question** and get a **probability** back, like `P(yes) = 0.96`.

## The data
6 realistic support tickets from an online shoe store (refunds, late orders, bugs, and more).

## What happens in this notebook
1. **Setup:** install the libraries, load the model, build the engine, and load the tickets.
2. **Look inside:** see the raw **logits** and which words the model wants to say next.
3. **Build `noul()`:** keep only "Yes" and "No" → softmax → P(yes).
4. **Checklist:** ask 4 yes/no questions about every ticket (refund? repeat problem? legal threat? card number?).
5. **Actions:** turn the numbers into real actions with thresholds (hide card data, alert legal, fast refund, or human check).
6. **Calibration:** find the best **temperature**, so the numbers are honest.

## Before you run
- ⚙️ Settings → **Accelerator: GPU** and **Internet: On**
- Then click **Run All**
- Look for `🔧 CHANGE` in the code. It shows what you can edit for your own use.

In [ ]:
import json, math, random
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# 🔧  CHANGE: MODEL_NAME
#   "Qwen/Qwen2.5-0.5B-Instruct" → faster, less smart (good for testing)
#   "Qwen/Qwen2.5-3B-Instruct"   → smarter, slower, needs more GPU memory
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"      # for batches: puts padding on the left, so every text ends at the last position
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 🔧 CHANGE: dtype=torch.bfloat16 → uses half the memory (needs a newer GPU, like L4 or A100)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
model.eval()
print("Loaded", MODEL_NAME, "on", device)

In [ ]:
# ===== THE ENGINE: all 3 methods are built on these small functions =====

# 🔧 CHANGE: SYSTEM_PROMPT → tell the model its job and its world.
#   Example: "You are a senior support agent at an online shoe store." → better domain judgment
SYSTEM_PROMPT = "You are a careful, fair judge. You answer with one short label only."

def format_state(state):
    """Text stays text. A dict becomes neat JSON, so the model sees every field and its name."""
    if isinstance(state, (dict, list)):
        return json.dumps(state, indent=2, ensure_ascii=False)
    return str(state)

def build_prompt(state, task):
    """Joins STATE + TASK (the question and the allowed answers) into the model's chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"STATE:\n{format_state(state)}\n\n{task}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 🔧 CHANGE: batch_size → bigger = faster. Make it smaller if you see "CUDA out of memory".
@torch.no_grad()
def next_token_logits(prompts, batch_size=8):
    """Runs the model on many prompts. Returns one row of logits per prompt: [n_prompts, vocab_size]."""
    rows = []
    for i in range(0, len(prompts), batch_size):
        batch = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        out = model(**batch, logits_to_keep=1)   # keep only the LAST position (saves memory). Old transformers? Remove it.
        rows.append(out.logits[:, -1, :].float().cpu())
    return torch.cat(rows)

def token_ids(forms):
    """Token ids for each spelling, with and without a space.
    Example: ["Yes", "yes"] → ids of "Yes", " Yes", "yes", " yes"."""
    ids = set()
    for f in forms:
        for v in (f, " " + f):
            t = tokenizer.encode(v, add_special_tokens=False)
            if len(t) == 1:
                ids.add(t[0])
    if not ids:
        raise ValueError(f"None of {forms} is one token. Use short answers like letters or digits.")
    return sorted(ids)

def answer_logits(row, answers):
    """answers = {"yes": ["Yes", "yes"], "no": ["No", "no"]}
    Joins all spellings of one answer into ONE logit (logsumexp = adds their probabilities together)."""
    return torch.stack([torch.logsumexp(row[token_ids(forms)], dim=0) for forms in answers.values()])

def answer_probs(row, answers, temperature=1.0):
    """Keep only our answers → divide by temperature → softmax → probabilities that add up to 1."""
    return torch.softmax(answer_logits(row, answers) / temperature, dim=0)

def confidence(probs):
    """1.0 = all probability on one answer. 0.0 = spread evenly. Based on entropy."""
    if len(probs) < 2:
        return 1.0
    entropy = -(probs * probs.clamp_min(1e-12).log()).sum().item()
    return 1 - entropy / math.log(len(probs))

print("Engine ready ✅")

In [ ]:
# Support tickets from an online shoe store.
# Each ticket is ONE STATE: everything the model needs to judge it.
# 🔧 CHANGE: put your own data here (emails, reviews, chats, CVs...).
#   Keep only the fields that help the decision. Extra fields = more noise and slower runs.
TICKETS = [
    {
        "ticket_id": "T-101",
        "customer": {"tier": "gold", "orders_last_year": 14},
        "order": {"item": "Trail running shoes", "price_usd": 120, "status": "delivered", "days_since_delivery": 3},
        "message": "Wrong size AGAIN. I ordered 40 and got 42, same as last month. I don't want another exchange, just give me my money back.",
    },
    {
        "ticket_id": "T-102",
        "customer": {"tier": "standard", "orders_last_year": 1},
        "order": {"item": "Leather boots", "price_usd": 210, "status": "in_transit", "days_since_order": 16},
        "message": "It's been 16 days. Tracking hasn't moved in a week. If it doesn't arrive by Friday I'm reporting you to consumer protection.",
    },
    {
        "ticket_id": "T-103",
        "customer": {"tier": "standard", "orders_last_year": 3},
        "order": None,
        "message": "Checkout keeps failing with error 502 when I press Pay. Tried two browsers. My card is 4532 7781 0092 4421 if that helps.",
    },
    {
        "ticket_id": "T-104",
        "customer": {"tier": "new", "orders_last_year": 0},
        "order": None,
        "message": "Hi! I run a running club with 30 members. Do you give a discount if we order 30 pairs together?",
    },
    {
        "ticket_id": "T-105",
        "customer": {"tier": "gold", "orders_last_year": 9},
        "order": {"item": "Sandals", "price_usd": 45, "status": "delivered", "days_since_delivery": 40},
        "message": "The sole came off after one month. Not happy. What can you do?",
    },
    {
        "ticket_id": "T-106",
        "customer": {"tier": "standard", "orders_last_year": 2},
        "order": {"item": "Sneakers", "price_usd": 85, "status": "delivered", "days_since_delivery": 1},
        "message": "I was charged twice for the same order, and the app crashes when I open my orders page.",
    },
]
print(len(TICKETS), "tickets loaded")

In [ ]:
# 👀 Look inside: what does the model want to say first?
task = "QUESTION: Is the customer asking for their money back?\nAnswer with only Yes or No."
row = next_token_logits([build_prompt(TICKETS[0], task)])[0]

print("Logits for one prompt:", row.shape[0])   # one score for every token the model knows
probs_all = torch.softmax(row, dim=0)            # softmax over ALL tokens
top = torch.topk(probs_all, 8)
for p, i in zip(top.values, top.indices):
    print(f"{tokenizer.decode(int(i))!r:10} logit={row[i].item():6.2f}  prob={p.item():.4f}")

In [ ]:
# 🔧 CHANGE: YES_NO for other languages → {"yes": ["Sí"], "no": ["No"]}, and change the prompt line too
YES_NO = {"yes": ["Yes", "yes"], "no": ["No", "no"]}

def noul_task(instructions, criteria=None):
    task = f"QUESTION: {instructions}\n"
    if criteria:   # optional: say exactly what counts as yes and as no
        task += f"Answer Yes if: {criteria['true']}\nAnswer No if: {criteria['false']}\n"
    return task + "Answer with only Yes or No."

def noul_margin(states, instructions, criteria=None):
    """logit(Yes) - logit(No) for each state. Positive = leans yes. Negative = leans no."""
    prompts = [build_prompt(s, noul_task(instructions, criteria)) for s in states]
    margins = []
    for row in next_token_logits(prompts):
        yes, no = answer_logits(row, YES_NO)
        margins.append((yes - no).item())
    return margins

def noul(states, instructions, criteria=None, temperature=1.0):
    """Returns P(yes) for each state (0 to 1). Always pass a LIST of states.
    Softmax over 2 answers = sigmoid(margin / temperature). Same math, simpler."""
    return [1 / (1 + math.exp(-m / temperature)) for m in noul_margin(states, instructions, criteria)]

p = noul([TICKETS[0]], "Is the customer asking for their money back?")
print(f"T-101 → P(yes) = {p[0]:.2f}")

In [ ]:
# A CHECKLIST: many yes/no questions, asked about EVERY ticket.
# 🔧 CHANGE: add or remove questions. One question = one thing.
#   Bad : "Is the customer angry and asking for a refund?"   (two things!)
#   Good: two separate questions → "is_angry" and "wants_refund"
CHECKLIST = {
    "wants_refund": {
        "instructions": "Is the customer asking for their money back?",
    },
    "repeat_problem": {
        "instructions": "Has this same problem happened to the customer before?",
        "criteria": {
            "true": "Says 'again', 'second time', 'same as last time', or mentions an earlier ticket",
            "false": "Nothing suggests an earlier problem",
        },
    },
    "legal_threat": {
        "instructions": "Does the customer threaten legal action or to report the company to an authority?",
    },
    "card_number": {
        "instructions": "Does the message contain a payment card number, or many digits of one?",
    },
}

table = pd.DataFrame({"ticket_id": [t["ticket_id"] for t in TICKETS]})
for name, q in CHECKLIST.items():
    table[name] = noul(TICKETS, q["instructions"], q.get("criteria"))
table.round(2)

In [ ]:
# Turn probabilities into ACTIONS with your own rules.
# 🔧 CHANGE: THRESHOLDS. Ask yourself: "What does a mistake cost?"
#   Missing a legal threat or a card number is very bad → LOW threshold (catch more, accept some false alarms)
#   A wrong refund costs money                          → HIGH threshold (act only when very sure)
THRESHOLDS = {"wants_refund": 0.8, "repeat_problem": 0.7, "legal_threat": 0.4, "card_number": 0.3}

# 🔧 CHANGE: UNSURE zone → these cases go to a human. Wider zone = more human work, fewer mistakes.
UNSURE = (0.3, 0.7)

def actions(row):
    out = []
    if row["card_number"] >= THRESHOLDS["card_number"]:
        out.append("🔒 hide card number")
    if row["legal_threat"] >= THRESHOLDS["legal_threat"]:
        out.append("⚖️ alert legal team")
    if row["wants_refund"] >= THRESHOLDS["wants_refund"] and row["repeat_problem"] >= THRESHOLDS["repeat_problem"]:
        out.append("💸 fast-track refund")
    elif UNSURE[0] < row["wants_refund"] < UNSURE[1]:
        out.append("🤔 human checks refund request")
    return ", ".join(out) or "normal queue"

table["action"] = table.apply(actions, axis=1)
table[["ticket_id", "action"]]

## 🎯 Calibration: make the numbers honest
Small models are often **too sure** or **not sure enough**.
If the model says 0.9, it should be right about 9 times out of 10.

**Fix:** try a few temperatures on examples where you KNOW the right answer, and keep the best one.
⚠️ 8 examples is only for learning. In real life, use 100 or more.

In [ ]:
# Examples with KNOWN answers. 1 = yes (wants money back), 0 = no.
# 🔧 CHANGE: use real messages from YOUR data, labeled by a person.
LABELED = [
    ("Please refund my order, the shoes fell apart.", 1),
    ("I want my money back.", 1),
    ("Cancel it and reverse the charge on my card.", 1),
    ("Return these and put the money back on my card, please.", 1),
    ("Where is my package?", 0),
    ("Can I exchange these for a bigger size?", 0),
    ("Do you sell kids' shoes?", 0),
    ("Love the shoes, thanks!", 0),
]
texts = [t for t, _ in LABELED]
labels = [y for _, y in LABELED]
margins = noul_margin(texts, CHECKLIST["wants_refund"]["instructions"])   # model runs only ONCE

def log_loss(T):
    """Lower = better. Gives a big penalty for being sure AND wrong."""
    total = 0
    for m, y in zip(margins, labels):
        p = min(max(1 / (1 + math.exp(-m / T)), 1e-6), 1 - 1e-6)
        total += -(y * math.log(p) + (1 - y) * math.log(1 - p))
    return total / len(labels)

TEMPS = [0.5, 1, 2, 3, 5]   # 🔧 CHANGE: temperatures to try
for T in TEMPS:
    print(f"T={T:<4} log loss = {log_loss(T):.3f}")

BEST_T = min(TEMPS, key=log_loss)
print("Best temperature:", BEST_T)

table["wants_refund_calibrated"] = noul(TICKETS, CHECKLIST["wants_refund"]["instructions"], temperature=BEST_T)
table[["ticket_id", "wants_refund", "wants_refund_calibrated"]].round(2)

## 💡 Tips
- Phrase it so that **high = yes**. Write "Contains card data?", not "Is free of card data?"
- Try the question **with and without criteria**. Keep whichever works better on your data.
- Set thresholds by **the cost of a mistake**, not by feeling.